In [ ]:
import sys, warnings, time
sys.path.append("..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
import tensorflow as tf
from tensorflow import keras  # type: ignore
from arch import arch_model

START = "2010-01-01"
N_SEEDS = 5
LOOK = 22
FOLD_START, FOLD_LEN = 0.6, 0.1           # fold 2 of notebook 10
REFIT_EVERY = 20
PRE_START = pd.Timestamp("2020-01-02")    # extra in-sample forecasts, baseline only
CRASH_START, CRASH_END = pd.Timestamp("2020-02-20"), pd.Timestamp("2020-04-30")

In [ ]:
_cache = {}
def load_data(ticker):
    if ticker not in _cache:
        px = yf.download(ticker, start=START, auto_adjust=True, progress=False)
        if isinstance(px.columns, pd.MultiIndex):
            px.columns = px.columns.get_level_values(0)
        px = px[["Open", "High", "Low", "Close"]].dropna()
        r = (100 * np.log(px["Close"] / px["Close"].shift(1))).dropna()
        r.name = "log_return"
        _cache[ticker] = (r, px)
    return _cache[ticker]

def range_proxy(r, px):
    lnHL = np.log(px["High"] / px["Low"])
    lnCO = np.log(px["Close"] / px["Open"])
    lnON = np.log(px["Open"] / px["Close"].shift(1))
    gk = 1e4 * (0.5 * lnHL ** 2 - (2 * np.log(2) - 1) * lnCO ** 2)
    gk_on = (gk + 1e4 * lnON ** 2).loc[r.index]
    sq_ret = r ** 2
    roll_scale = (sq_ret.rolling(252, min_periods=126).sum()
                  / gk_on.rolling(252, min_periods=126).sum()).shift(1).bfill()
    return (roll_scale * gk_on).clip(lower=1e-6)

SPECS = {
    "GARCH(1,1)": dict(vol="GARCH", p=1, q=1),
    "EGARCH":     dict(vol="EGARCH", p=1, o=1, q=1),
    "GJR-GARCH":  dict(vol="GARCH", p=1, o=1, q=1),
}

def garch_forecasts(r, split, end, pre):
    """Estimate on r[:split] only; forecasts for r[pre:end] (pre < split gives in-sample baseline days)."""
    gf_all, test = {}, {}
    for name, spec in SPECS.items():
        res = arch_model(r, mean="Constant", dist="t", **spec).fit(last_obs=r.index[split - 1], disp="off")
        gf = res.forecast(horizon=1, start=r.index[0], reindex=False).variance["h.1"]
        gf_all[name] = gf
        test[name] = gf.shift(1).loc[r.index[pre:end]]
    test["Rolling 21d var"] = r.rolling(21).var().shift(1).loc[r.index[pre:end]]
    return gf_all, pd.DataFrame(test)

def har_design(x):
    return pd.DataFrame({"daily": x, "weekly": x.rolling(5).mean(), "monthly": x.rolling(22).mean()})

def rolling_har(series, split, end, pre, log=False):
    x = np.log(series) if log else series
    X = sm.add_constant(har_design(x))
    y_next = x.shift(-1)
    preds = []
    for s in range(split, end, REFIT_EVERY):
        e = min(s + REFIT_EVERY, end)
        Xt, yt = X.iloc[: s - 1], y_next.iloc[: s - 1]
        ok = Xt.notna().all(axis=1) & yt.notna()
        res = sm.OLS(yt[ok], Xt[ok]).fit()
        a = pre if s == split else s          # first block also forecasts the in-sample days
        p = res.predict(X.iloc[a - 1: e - 1])
        if log:
            p = np.exp(p + 0.5 * res.mse_resid)
        preds.append(pd.Series(p.values, index=series.index[a:e]))
    return pd.concat(preds)

In [ ]:
def qlike_loss(y_true, y_pred):
    y_, base = y_true[:, 0:1], y_true[:, 1:2]
    out = tf.clip_by_value(y_pred, -4.0, 4.0)
    f = base * tf.exp(out)
    return tf.reduce_mean(tf.math.log(f) + y_ / f)

def build_model(n_feat, lr=3e-4):
    m = keras.Sequential([
        keras.layers.Input(shape=(LOOK, n_feat)),
        keras.layers.LSTM(16),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, kernel_initializer="zeros", bias_initializer="zeros"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(lr), loss=qlike_loss)
    return m

def train_ensemble(s, n_seeds):
    """Returns (ensemble-mean forecast, per-seed clipped log-multipliers [seeds x days])."""
    preds, outs = [], []
    for seed in range(n_seeds):
        np.random.seed(seed); tf.random.set_seed(seed)
        m = build_model(s["X_tr"].shape[2])
        early = keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
        m.fit(s["X_tr"], s["Y_tr"], validation_data=(s["X_val"], s["Y_val"]),
              epochs=200, batch_size=64, callbacks=[early], verbose=0)
        out = np.clip(m.predict(s["X_te"], verbose=0).ravel(), -4, 4)
        outs.append(out)
        preds.append(s["Y_te"][:, 1] * np.exp(out))
    return np.array(preds).mean(axis=0), np.array(outs)

def nn_forecasts(r, gjr_gf, split, end, n_seeds, pre_date):
    feat = pd.DataFrame({
        "ret": r, "sq_ret": r ** 2,
        "abs_mean_5d": r.abs().rolling(5).mean(),
        "rv_21d": r.rolling(21).std() * np.sqrt(252),
        "log_garch_var": np.log(gjr_gf),
    }).dropna()
    split_date = r.index[split]
    end_date = r.index[end] if end < len(r) else r.index[-1] + pd.Timedelta(days=1)

    tr = feat.index < split_date
    mu, sd = feat[tr].mean(), feat[tr].std()
    feat_s = ((feat - mu) / sd).clip(-5, 5)
    vals, idx = feat_s.values, feat_s.index
    rsq = (r ** 2).loc[idx].values
    gv = gjr_gf.loc[idx].values

    X, y, g, dates = [], [], [], []
    for t in range(LOOK, len(idx)):
        X.append(vals[t - LOOK:t]); y.append(rsq[t]); g.append(gv[t - 1]); dates.append(idx[t])
    X, y, g, dates = np.array(X), np.array(y), np.array(g), pd.DatetimeIndex(dates)

    train_mask = dates < split_date
    test_mask = (dates >= pre_date) & (dates < end_date)     # includes in-sample baseline days
    const_var = (r ** 2)[r.index < split_date].mean()

    def make_sets(X_, base):
        Y = np.column_stack([y, base])
        Xa, Ya = X_[train_mask], Y[train_mask]
        v = int(len(Xa) * 0.9)
        return dict(X_tr=Xa[:v], Y_tr=Ya[:v], X_val=Xa[v:], Y_val=Ya[v:],
                    X_te=X_[test_mask], Y_te=Y[test_mask])

    out = pd.DataFrame(index=dates[test_mask])
    out["Hybrid"], hyb_raw = train_ensemble(make_sets(X, g), n_seeds)
    out["LSTM"], _ = train_ensemble(make_sets(X[:, :, :4], np.full(len(y), const_var)), n_seeds)
    return out, pd.DataFrame(hyb_raw.T, index=dates[test_mask]), feat, mu, sd, split_date

In [ ]:
r, px = load_data("^GSPC")
proxy_rb = range_proxy(r, px)
n = len(r)
split = int(n * FOLD_START)
end = min(int(n * (FOLD_START + FOLD_LEN)), n)       # identical expression to notebook 10
pre = r.index.searchsorted(PRE_START)
assert r.index[split] == pd.Timestamp("2020-02-10"), r.index[split]
print("Fold 2:", r.index[split].date(), "->", r.index[end - 1].date(), f"({end - split} days)")

t0 = time.time()
gf_all, fc = garch_forecasts(r, split, end, pre)
nn, hyb_raw, feat, mu, sd, split_date = nn_forecasts(r, gf_all["GJR-GARCH"], split, end, N_SEEDS, PRE_START)
floor = 0.01 * proxy_rb.iloc[:split].mean()
har = rolling_har(proxy_rb, split, end, pre).clip(lower=floor).rename("HAR")
loghar = rolling_har(proxy_rb, split, end, pre, log=True).rename("log-HAR")
fc_all = fc.join(nn, how="inner").join(har, how="inner").join(loghar, how="inner").dropna().clip(lower=1e-8)
fc = fc_all.loc[fc_all.index >= split_date]           # the real fold-2 evaluation set
print(f"retrained in {time.time() - t0:.0f}s; fold days: {len(fc)}; incl. in-sample baseline days: {len(fc_all)}")

MODELS = list(fc.columns)
PROXIES = {"Squared returns": r ** 2, "Range-based": proxy_rb}
loss = {pn: fc.apply(lambda f: np.log(f) + p.loc[fc.index] / f) for pn, p in PROXIES.items()}
print("\nSanity check - fold-2 QLIKE (compare with notebook 10's fold 2 row):")
display(pd.DataFrame({pn: L.mean() for pn, L in loss.items()}).round(4))

phase = pd.Series("recovery", index=fc.index)
phase[fc.index < CRASH_START] = "pre-crash"
phase[(fc.index >= CRASH_START) & (fc.index <= CRASH_END)] = "crash"
PHASES = ["pre-crash", "crash", "recovery"]
print(phase.value_counts()[PHASES].to_dict())

## 1. Forecast paths, Feb 1 - Jun 30 2020 (log scale)
Shaded area = before fold 2 starts (in-sample forecasts).

In [ ]:
w = slice("2020-02-01", "2020-06-30")
groups = {"GARCH family and rolling variance": ["GARCH(1,1)", "EGARCH", "GJR-GARCH", "Rolling 21d var"],
          "HAR and neural networks (GJR-GARCH for reference)": ["HAR", "log-HAR", "LSTM", "Hybrid", "GJR-GARCH"]}
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for ax, (title, ms) in zip(axes, groups.items()):
    ax.plot(r.loc[w].index, (r ** 2).loc[w], ".", color="0.6", ms=4, label="Squared return")
    ax.plot(proxy_rb.loc[w].index, proxy_rb.loc[w], "-", color="black", lw=1.2, label="Range proxy")
    for m in ms:
        ax.plot(fc_all.loc[w].index, fc_all.loc[w, m], lw=1.5, label=m, ls="--" if m == "GJR-GARCH" and len(ms) == 5 else "-")
    ax.axvspan(pd.Timestamp("2020-02-01"), split_date, color="0.9", zorder=0)
    ax.axvline(CRASH_START, color="red", lw=0.8, ls=":")
    ax.set_yscale("log"); ax.set_ylabel("variance (%^2)"); ax.set_title(title); ax.legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.show()

## 2. Reaction speed
Baseline = mean forecast in Jan 2020 (in-sample for the models). Days are trading days counted from Feb 20 (= day 0), searched through Jun 30. Peak = within the crash phase (Feb 20 - Apr 30).

In [ ]:
base = fc_all.loc["2020-01-01":"2020-01-31"].mean()
look = fc_all.loc["2020-02-20":"2020-06-30"]
cw = fc_all.loc[CRASH_START:CRASH_END]
def first_cross(s, level):
    hit = np.flatnonzero(s.values >= level)
    return int(hit[0]) if len(hit) else np.nan
rows = {}
for m in MODELS:
    rows[m] = {"Jan-2020 level": base[m],
               "days to 2x": first_cross(look[m], 2 * base[m]),
               "days to 5x": first_cross(look[m], 5 * base[m]),
               "peak forecast": cw[m].max(), "peak date": cw[m].idxmax().date(),
               "peak / baseline": cw[m].max() / base[m]}
react = pd.DataFrame(rows).T
display(react.round(3))
ref = {}
for pn, p in PROXIES.items():
    b = p.loc["2020-01-01":"2020-01-31"].mean(); pk = p.loc[CRASH_START:CRASH_END]
    ref[pn] = {"Jan-2020 mean": b, "peak": pk.max(), "peak date": pk.idxmax().date(), "peak / Jan mean": pk.max() / b}
print("Proxy peaks for comparison:")
display(pd.DataFrame(ref).T.round(3))

## 3. Where the losses come from
Phases: pre-crash (to Feb 19), crash (Feb 20 - Apr 30), recovery (May 1 onward).

In [ ]:
by_phase = {}
for pn, L in loss.items():
    t = L.groupby(phase).mean().T[PHASES]
    by_phase[pn] = t
    print(f"QLIKE by phase - {pn} proxy")
    display(t.round(4))
print("Days per phase:", phase.value_counts()[PHASES].to_dict())

# Contribution to the fold-level gap: sum of (model - GJR) loss per phase, divided by fold length
for pn, L in loss.items():
    d = L.sub(L["GJR-GARCH"], axis=0).groupby(phase).sum().T[PHASES] / len(L)
    d["total"] = d.sum(axis=1)
    print(f"Contribution to fold-average QLIKE gap vs GJR-GARCH - {pn} (negative = better than GJR)")
    display(d.drop("GJR-GARCH").round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, (pn, L) in zip(axes, loss.items()):
    cum = L.sub(L["GJR-GARCH"], axis=0).cumsum()
    for m in MODELS:
        if m != "GJR-GARCH":
            ax.plot(cum.index, cum[m], label=m, lw=1.4)
    ax.axhline(0, color="black", lw=0.8); ax.axvspan(CRASH_START, CRASH_END, color="red", alpha=0.08)
    ax.set_title(f"Cumulative QLIKE minus GJR-GARCH ({pn})"); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

## 4. Under- vs over-prediction
Mean of log(proxy / forecast); positive = the model under-predicted. A perfectly calibrated forecast gives about -1.27 for squared returns under normal errors (E[log chi-square(1)]) and a value nearer 0 for the less noisy range proxy, so compare models with each other and across phases rather than against zero.

In [ ]:
for pn, p in PROXIES.items():
    lr_ = np.log(p.loc[fc.index].values[:, None] / fc.values)
    t = pd.DataFrame(lr_, index=fc.index, columns=MODELS).groupby(phase).mean().T[PHASES]
    print(f"Mean log(proxy / forecast) - {pn}")
    display(t.round(3))

## 5. Hybrid internals
The network outputs a log-multiplier (clipped to +/-4) on the GJR-GARCH forecast. Ensemble multiplier = log(Hybrid forecast / GJR forecast); per-seed values are the raw clipped network outputs.

In [ ]:
ens_lm = np.log(fc["Hybrid"] / fc["GJR-GARCH"])
seed_lm = hyb_raw.loc[fc.index]
at_clip = (seed_lm.abs() >= 4 - 1e-6)

fig, ax = plt.subplots(figsize=(13, 4.5))
for c in seed_lm.columns:
    ax.plot(seed_lm.index, seed_lm[c], lw=0.5, color="0.7")
ax.plot(ens_lm.index, ens_lm, color="C3", lw=1.5, label="ensemble log(Hybrid / GJR)")
ax.axhline(0, color="black", lw=0.8); ax.axhline(4, ls=":", color="k"); ax.axhline(-4, ls=":", color="k")
ax.axvspan(CRASH_START, CRASH_END, color="red", alpha=0.08)
ax.set_ylabel("log-multiplier"); ax.set_title("Hybrid log-multiplier on the GJR-GARCH forecast (grey = individual seeds)")
ax.legend(); plt.tight_layout(); plt.show()

summ = pd.DataFrame({
    "mean ensemble log-mult": ens_lm.groupby(phase).mean(),
    "share of days ensemble > 0": (ens_lm > 0).groupby(phase).mean(),
    "share of seed-days at +4 clip": (seed_lm >= 4 - 1e-6).mean(axis=1).groupby(phase).mean(),
    "share of seed-days at -4 clip": (seed_lm <= -4 + 1e-6).mean(axis=1).groupby(phase).mean(),
}).loc[PHASES]
display(summ.round(3))
print(f"Whole fold: share of seed-days at the +/-4 clip = {at_clip.values.mean():.1%}"
      f" (+4: {(seed_lm >= 4 - 1e-6).values.mean():.1%}, -4: {(seed_lm <= -4 + 1e-6).values.mean():.1%})")
c = ens_lm[phase == "crash"]
print(f"During the crash the network {'RAISED' if c.mean() > 0 else 'LOWERED'} the GJR forecast on average "
      f"(mean log-multiplier {c.mean():+.2f}, i.e. x{np.exp(c.mean()):.2f}); raised on {(c > 0).mean():.0%} of crash days.")

## 6. Input extrapolation
Features are z-scored with training-period mean/std (then clipped to +/-5 before entering the network). Below, z-scores are computed *before* the clip. Crash days = feature dates Feb 20 - Apr 30 2020. The LSTM uses the first four features; the hybrid also uses log_garch_var.

In [ ]:
z = (feat - mu) / sd
ztr = z[feat.index < split_date]
zc = z.loc[CRASH_START:CRASH_END]
nets = {"ret": "LSTM + Hybrid", "sq_ret": "LSTM + Hybrid", "abs_mean_5d": "LSTM + Hybrid",
        "rv_21d": "LSTM + Hybrid", "log_garch_var": "Hybrid only"}
rows = {}
for f_ in z.columns:
    beyond_hi = zc[f_] > ztr[f_].max()
    beyond_lo = zc[f_] < ztr[f_].min()
    rows[f_] = {"used by": nets[f_],
                "train max (z)": ztr[f_].max(), "train min (z)": ztr[f_].min(),
                "crash max (z)": zc[f_].max(), "crash min (z)": zc[f_].min(),
                "crash days > train max": int(beyond_hi.sum()), "crash days < train min": int(beyond_lo.sum()),
                "max excess above train max (SD)": max(zc[f_].max() - ztr[f_].max(), 0),
                "crash days beyond +/-5 (clipped)": int((zc[f_].abs() > 5).sum())}
ext = pd.DataFrame(rows).T
display(ext.round(2))
nc = len(zc)
for name, cols in {"LSTM (4 features)": list(z.columns[:4]), "Hybrid (5 features)": list(z.columns)}.items():
    any_beyond = (zc[cols] > ztr[cols].max()).any(axis=1)
    print(f"{name}: {int(any_beyond.sum())} of {nc} crash days have at least one feature above its training maximum")